# Atividade: construindo um classificador de frutas

Você receberá exemplos de maçãs, tangerinas, laranjas e limões. Sua tarefa é criar, por conta própria, uma função que tente reconhecer cada fruta usando `if`, `elif` e `else`.

Não existe um roteiro indicando quais características ou intervalos usar. Parte da atividade é descobrir uma estratégia, perceber os casos difíceis e decidir como lidar com eles. Execute as células na ordem e não procure o gabarito do teste antes de congelar suas regras. Ao longo do notebook, usaremos nomes comuns em aprendizado de máquina — como `train`, `test`, `X`, `y`, `fit` e `predict` — e construiremos o significado de cada um.

## 1. Conhecer os exemplos disponíveis

O código abaixo lê um arquivo CSV com Pandas. Cada linha representa uma fruta. As colunas `mass`, `width`, `height` e `color_score` descrevem a fruta; `fruit_name` informa seu nome. Nos dados originais, tangerina aparece como `mandarin`.

In [1]:
import pandas as pd

TRAIN_URL = 'https://raw.githubusercontent.com/H2IA/treinamento/main/data/frutas/train.csv'
train = pd.read_csv(TRAIN_URL)

FEATURES = ['mass', 'width', 'height', 'color_score']
TARGET = 'fruit_name'

X_train = train[FEATURES]
y_train = train[TARGET]

pd.set_option('display.max_rows', None)
display(train)

,id,mass,width,height,color_score,fruit_name
0,F001,192,8.4,7.3,0.55,apple
1,F002,180,8.0,6.8,0.59,apple
2,F003,86,6.2,4.7,0.80,mandarin
3,F004,84,6.0,4.6,0.79,mandarin
4,F005,80,5.8,4.3,0.77,mandarin
5,F006,80,5.9,4.3,0.81,mandarin
6,F007,178,7.1,7.8,0.92,apple
7,F008,172,7.4,7.0,0.89,apple
8,F009,172,7.1,7.6,0.92,apple
9,F010,154,7.0,7.1,0.88,apple


Examine todas as linhas antes de programar. Procure relações entre os valores e o nome da fruta, experimente hipóteses e anote o que considerar importante. O notebook não calculará intervalos nem indicará quais características usar: decidir isso faz parte do problema.

In [2]:
# vendo os mínimos, médias e máximos de cada fruta pra achar padrões
train.groupby('fruit_name')[FEATURES].agg(['min', 'mean', 'max'])

mass                  width                height                  \
            min        mean  max   min      mean  max    min      mean   max   
fruit_name                                                                     
apple       140  163.733333  192   7.0  7.486667  8.4    6.8  7.300000   7.8   
lemon       116  149.538462  216   5.9  6.553846  7.3    7.5  8.807692  10.5   
mandarin     80   82.500000   86   5.8  5.975000  6.2    4.3  4.475000   4.7   
orange      140  187.875000  356   6.7  7.481250  9.2    7.1  7.918750   9.4   

           color_score                  
                   min      mean   max  
fruit_name                              
apple             0.55  0.783333  0.92  
lemon             0.70  0.719231  0.74  
mandarin          0.77  0.792500  0.81  
orange            0.72  0.772500  0.82

In [3]:
# olhando as maçãs e laranjas ordenadas pelo color_score
train[train['fruit_name'].isin(['apple', 'orange'])].sort_values('color_score')[['id', 'mass', 'width', 'height', 'color_score', 'fruit_name']]

,id,mass,width,height,color_score,fruit_name
0,F001,192,8.4,7.3,0.55,apple
1,F002,180,8.0,6.8,0.59,apple
13,F014,156,7.6,7.5,0.67,apple
11,F012,152,7.6,7.3,0.69,apple
12,F013,156,7.7,7.1,0.69,apple
10,F011,164,7.3,7.7,0.70,apple
22,F023,140,6.7,7.1,0.72,orange
26,F027,190,7.5,8.1,0.74,orange
20,F021,356,9.2,9.2,0.75,orange
27,F028,142,7.6,7.8,0.75,orange


Quando tiver uma primeira ideia, registre-a com suas palavras. Não é necessário encontrar uma separação perfeita antes de começar a programar.

**Minha estratégia inicial:** olhando a tabela, dá pra ver de cara que as tangerinas (`mandarin`) são muito mais leves e baixinhas que todo o resto (`mass < 100`). Os limões (`lemon`) são sempre mais compridos que largos (`height >= 7.5` e `width <= 7.3`) e têm `color_score` baixo (`<= 0.74`). Já entre maçã (`apple`) e laranja (`orange`), o peso e tamanho se misturam bastante, mas o `color_score` separa super bem: as laranjas ficam todas concentradas no meio (`0.72` a `0.82`), enquanto as maçãs ou são mais esverdeadas/amareladas (`<= 0.70`) ou bem vermelhas (`>= 0.83`).

## 2. Escrever as regras

Em aprendizado de máquina, costumamos chamar as entradas de `X` e as respostas conhecidas de `y`. Por isso, separamos acima `X_train` e `y_train`: são as entradas e respostas disponíveis durante o ajuste.

Mais adiante, uma biblioteca poderá aprender automaticamente com uma chamada como `model.fit(X_train, y_train)` e gerar respostas com `model.predict(X_test)`. Nesta atividade, você fará o papel do `fit` manualmente: observará os exemplos e decidirá as condições da função `predict_fruit`.

Complete a função abaixo. Ela deve devolver exatamente um destes nomes: `apple`, `mandarin`, `orange` ou `lemon`. Você pode criar quantas condições forem necessárias, mas deve conseguir explicar cada uma delas.

In [4]:
def predict_fruit(mass, width, height, color_score):
    if mass < 100:
        return 'mandarin'
    elif height >= 7.5 and width <= 7.3 and color_score <= 0.74:
        return 'lemon'
    elif color_score <= 0.70 or color_score >= 0.83:
        return 'apple'
    else:
        return 'orange'

# Teste manualmente uma fruta antes de aplicar a função a todas as linhas.
predict_fruit(mass=192, width=8.4, height=7.3, color_score=0.55)

'apple'

In [5]:
def predict(X):
    return X.apply(
        lambda row: predict_fruit(
            row['mass'],
            row['width'],
            row['height'],
            row['color_score'],
        ),
        axis=1,
    )

train_results = train[['id'] + FEATURES].copy()
train_results['y_true'] = y_train
train_results['y_pred'] = predict(X_train)
train_results['correct'] = train_results['y_pred'] == train_results['y_true']
train_results.head()

,id,mass,width,height,color_score,y_true,y_pred,correct
0,F001,192,8.4,7.3,0.55,apple,apple,True
1,F002,180,8.0,6.8,0.59,apple,apple,True
2,F003,86,6.2,4.7,0.80,mandarin,mandarin,True
3,F004,84,6.0,4.6,0.79,mandarin,mandarin,True
4,F005,80,5.8,4.3,0.77,mandarin,mandarin,True


In [6]:
train_accuracy = train_results['correct'].mean()
print(f'Train accuracy: {train_accuracy:.1%}')
display(train_results.loc[~train_results['correct']])

Train accuracy: 97.9%


,id,mass,width,height,color_score,y_true,y_pred,correct
10,F011,164,7.3,7.7,0.7,apple,lemon,False


Agora você pode revisar livremente suas regras olhando apenas para `train`. Cada tentativa pode corrigir alguns casos e criar novos erros. Continue até chegar a uma solução que considere razoável e que consiga explicar. Depois registre:

- **qual resultado você conseguiu em `train`:** consegui 97.9% de acurácia (47 acertos em 48 frutas).
- **quais casos continuaram difíceis:** a única fruta que errou no treino foi a F011 (uma maçã com width = 7.3, height = 7.7 e color_score = 0.70), que acabou caindo na regra do limão por ser um pouco mais alta do que larga e ter cor baixa. Além disso, separar maçãs e laranjas usando peso ou largura é difícil porque os valores se sobrepõem bastante.
- **por que decidiu encerrar os ajustes nessa versão:** decidi parar por aqui porque as 4 condições ficaram simples e fáceis de justificar. Se eu colocasse mais um if específico só pra forçar o acerto da F011, estaria basicamente decorando os dados de treino (overfitting) em vez de criar uma regra geral.

## 3. Congelar as regras

Antes de abrir os exemplos novos:

1. anote sua acurácia nos exemplos conhecidos: 97.9% (47/48)
2. estime quantas das 11 frutas novas sua função acertará;
3. registre uma justificativa curta;
4. salve esta versão do notebook e não altere mais a função até avaliar o teste.

**Minha estimativa:** 9 ou 10 acertos em 11.

**Minha justificativa:** como as regras separaram muito bem as 4 frutas no treino (97.9%) usando características bem marcantes (peso da tangerina, formato alongado do limão e faixas de cor entre maçã e laranja), acho que vai acertar quase todas no teste, podendo errar 1 ou 2 casos se aparecer alguma maçã ou laranja com color_score bem no limite entre as faixas.

## 4. Fazer previsões para frutas novas

O próximo arquivo não contém `fruit_name`. Gere as previsões sem alterar as regras.

In [7]:
TEST_URL = 'https://raw.githubusercontent.com/H2IA/treinamento/main/data/frutas/test.csv'
test = pd.read_csv(TEST_URL)
X_test = test[FEATURES]
test

,id,mass,width,height,color_score
0,T01,176,7.4,7.2,0.60
1,T02,76,5.8,4.0,0.81
2,T03,166,6.9,7.3,0.93
3,T04,168,7.5,7.6,0.73
4,T05,170,7.6,7.9,0.88
5,T06,362,9.6,9.2,0.74
6,T07,164,7.2,7.0,0.80
7,T08,150,7.1,7.9,0.75
8,T09,194,7.2,10.3,0.70
9,T10,132,5.8,8.7,0.73


In [8]:
test_predictions = test[['id']].copy()
test_predictions['prediction'] = predict(X_test)
test_predictions.to_csv('fruit_predictions.csv', index=False)
display(test_predictions)

try:
    from google.colab import files
    files.download('fruit_predictions.csv')
except ImportError:
    print('The file fruit_predictions.csv was saved in the current folder.')

,id,prediction
0,T01,apple
1,T02,mandarin
2,T03,apple
3,T04,orange
4,T05,apple
5,T06,orange
6,T07,orange
7,T08,orange
8,T09,lemon
9,T10,lemon


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. O que você acabou de fazer?

Agora podemos dar nome às partes da experiência:

- massa, largura, altura e medida de cor são as **características** (*features*);
- o nome da fruta é a **classe** ou **rótulo** (*label*);
- `X` reúne as **características** (*features*) usadas como entrada;
- `y` reúne a **classe** ou **rótulo** (*label*) que queremos prever;
- sua função de regras é um **modelo**: `predict(X)` transforma características em previsões;
- `X_train` e `y_train` formam os **dados de treino**;
- você fez o papel de `fit` manualmente, ajustando o modelo com esses dados;
- `X_test` contém os **dados de teste**, que o modelo ainda não tinha visto;
- funcionar em exemplos novos é o que chamamos de **generalização**.

Você ainda não treinou automaticamente um algoritmo de aprendizado de máquina. A atividade separou as etapas para que fosse possível sentir a dificuldade de formular regras e avaliar se elas generalizam.

## 6. Avaliar depois da liberação do gabarito

Cole abaixo o link direto fornecido pelo professor. Execute esta seção somente depois de entregar ou congelar suas previsões.

In [9]:
ANSWER_KEY_URL = 'test_labels.csv'  # Arquivo liberado pelo professor.

if not ANSWER_KEY_URL.strip():
    print('O gabarito ainda não foi informado. Preserve suas previsões e volte depois.')
else:
    answer_key = pd.read_csv(ANSWER_KEY_URL)
    test_results = test_predictions.merge(answer_key, on='id', validate='one_to_one')
    test_results = test_results.rename(
        columns={'fruit_name': 'y_true', 'prediction': 'y_pred'}
    )
    test_results['correct'] = test_results['y_pred'] == test_results['y_true']
    test_accuracy = test_results['correct'].mean()

    print(f'Test accuracy: {test_accuracy:.1%}')
    print('\nAcurácia por classe:')
    accuracy_by_class = test_results.groupby('y_true')['correct'].mean()
    display(accuracy_by_class.rename('accuracy'))

    print('Matriz de confusão:')
    confusion_matrix = pd.crosstab(
        test_results['y_true'],
        test_results['y_pred'],
        rownames=['y_true'],
        colnames=['y_pred'],
        dropna=False,
    )
    display(confusion_matrix)

    print('Erros:')
    display(test_results.loc[~test_results['correct']])

Test accuracy: 90.9%

Acurácia por classe:


y_true
apple       0.75
lemon       1.00
mandarin    1.00
orange      1.00
Name: accuracy, dtype: float64


Matriz de confusão:


y_pred    apple  lemon  mandarin  orange
y_true                                  
apple         3      0         0       1
lemon         0      3         0       0
mandarin      0      0         1       0
orange        0      0         0       3


Erros:


    id  y_pred y_true  correct
3  T04  orange  apple    False


## 7. Fechamento

Responda com suas palavras:

1. **O resultado do teste ficou acima ou abaixo de sua estimativa?**
   ficou exatamente dentro da minha estimativa! Eu tinha estimado acertar 9 ou 10 das 11 frutas e o modelo acertou 10 de 11 (90.9%).

2. **Compare a acurácia nos dados de treino com a acurácia no teste. O que mudou?**
   a acurácia caiu um pouco: foi de 97.9% (47/48) no treino para 90.9% (10/11) no teste. Isso é super normal, já que as regras foram criadas olhando pros exemplos do treino e, quando chegam dados novos, podem aparecer valores um pouco diferentes daqueles limites exatos.

3. **Qual classe teve o pior resultado? A acurácia total deixava isso evidente?**
   a classe `apple` (maçã) teve o pior resultado no teste, com 75% de acerto (3 de 4), enquanto `lemon`, `mandarin` e `orange` tiveram 100%. Olhando só para a acurácia total (90.9%), não ficava evidente que o erro tinha acontecido só nas maçãs.

4. **Observe os erros. Há frutas com valores fora dos intervalos vistos no treino?**
   sim! A única fruta que o modelo errou no teste foi a `T04` (que era `apple` mas foi prevista como `orange`), porque ela tem `color_score = 0.73`. No treino, as maçãs mais claras iam só até `0.70` (e as vermelhas começavam em `0.83`), então esse `0.73` caiu justo na faixa onde só existiam laranjas no treino. Além dela, outras frutas do teste também tinham valores fora dos mínimos/máximos do treino, como a `T02` (`mass = 76` e `height = 4.0`), `T03` (`width = 6.9` e `color_score = 0.93`), `T06` (`mass = 362` e `width = 9.6`) e `T10` (`width = 5.8`).

5. **O que você mudaria nas regras? Por que não devemos usar o mesmo teste repetidamente para fazer esses ajustes?**
   eu tentaria combinar o `color_score` com outra característica (como a relação entre largura e altura ou a massa) para não depender só do corte seco em `color_score <= 0.70` para separar maçãs claras de laranjas. Não devemos usar o mesmo teste várias vezes para ajustar as regras porque isso vaza informação do teste para o modelo: acabaríamos decorando os casos do teste e ele deixaria de servir para medir a generalização em dados novos.

6. **Que vantagem um algoritmo de treinamento poderia ter sobre o ajuste manual? Que cuidado ainda continuaria necessário?**
   um algoritmo consegue calcular e testar automaticamente os melhores pontos de corte e combinações de variáveis muito mais rápido do que fazer tudo no olho na tentativa e erro (principalmente quando existem muitas colunas). Mesmo assim, o cuidado de separar treino e teste e evitar que o modelo decore os dados (*overfitting*) continua sendo indispensável.